In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl")
sys.path.insert(0, str(ROOT))

In [2]:
from moduly.db_core import utworz_engine_do_db
from sqlalchemy import text

In [3]:
silnik = utworz_engine_do_db()

In [29]:
misja_id = 8831
fabula_en = 'Ripple Effects'

In [27]:
q_select_kolejnosc_misja = text("""
SELECT KOLEJNOSC_LINII_FABULARNEJ
FROM MISJE
WHERE KOLEJNOSC_LINII_FABULARNEJ IS NOT NULL
  AND NAZWA_LINII_FABULARNEJ_EN = :fabula_en
  AND MISJA_ID_MOJE_PK = :misja_id
ORDER BY KOLEJNOSC_LINII_FABULARNEJ ASC
""")

In [28]:
q_select_podsumowanie = text("""
SELECT 
	M.KOLEJNOSC_LINII_FABULARNEJ AS NUMER_MISJI_W_CHAINIE, 
	MP.PODSUMOWANIE
FROM MISJE AS M
INNER JOIN MISJE_PODSUMOWANIA AS MP
  ON M.MISJA_ID_MOJE_PK = MP.MISJA_ID_MOJE_FK
WHERE KOLEJNOSC_LINII_FABULARNEJ IS NOT NULL
  AND M.NAZWA_LINII_FABULARNEJ_EN = :fabula_en
  AND KOLEJNOSC_LINII_FABULARNEJ <= :kolejnosc_misji
ORDER BY NUMER_MISJI_W_CHAINIE ASC
""")

In [33]:
with silnik.connect() as conn:
    kolejnosc_misja = conn.execute(q_select_kolejnosc_misja, {
                                                            "misja_id": misja_id,
                                                            "fabula_en": fabula_en
                                                        }).first()

    if kolejnosc_misja:
        kolejnosc_misja = kolejnosc_misja[0]

        podsumowanie_txt = conn.execute(q_select_podsumowanie, {
                                                                "misja_id": misja_id,
                                                                "fabula_en": fabula_en,
                                                                "kolejnosc_misji": kolejnosc_misja
                                                            }).all()
    

In [34]:
podsumowanie_txt

[(1, "Silvermoon otrzymuje pilne wieści: Twilight's Blade okazuje się większy i silniejszy, niż przypuszczano. Magister Umbric próbuje dostać się do Voidst ... (121 characters truncated) ... cie pożera wioskę. Lady Liadrin opuszcza miejsce, więc trzeba ją dogonić przy Suncrown's Southern gate i sprowadzić ocalałych, zanim będzie za późno."),
 (2, "Lady Liadrin i inni oglądają Suncrown Village pochłonięte przez Lightbloom. Rutaani przyspieszają rozprzestrzenianie, jak wcześniej w Fairbreeze, a A ... (85 characters truncated) ... gwałtowny wzrost i poleca się zająć sprawą. Gdy Farstriders badają hałas, gracz próbuje ograniczyć Lightbloom w murach i wykasowuje tyle, ile się da."),
 (3, "W Suncrown Village Lady Liadrin opowiada, że Runestone Shan'dor powstrzymuje rozrastający się Lightbloom, lecz kolejne uderzenie może przekreślić nad ... (72 characters truncated) ...  strony. Zadaniem jest ich odnaleźć, osłonić i poprowadzić w bezpieczne miejsce przy Runestone, dopóki możliwe jest skutecz

In [35]:
set(n for n in podsumowanie_txt)

{(1, "Silvermoon otrzymuje pilne wieści: Twilight's Blade okazuje się większy i silniejszy, niż przypuszczano. Magister Umbric próbuje dostać się do Voidst ... (121 characters truncated) ... cie pożera wioskę. Lady Liadrin opuszcza miejsce, więc trzeba ją dogonić przy Suncrown's Southern gate i sprowadzić ocalałych, zanim będzie za późno."),
 (2, "Lady Liadrin i inni oglądają Suncrown Village pochłonięte przez Lightbloom. Rutaani przyspieszają rozprzestrzenianie, jak wcześniej w Fairbreeze, a A ... (85 characters truncated) ... gwałtowny wzrost i poleca się zająć sprawą. Gdy Farstriders badają hałas, gracz próbuje ograniczyć Lightbloom w murach i wykasowuje tyle, ile się da."),
 (3, "W Suncrown Village Lady Liadrin opowiada, że Runestone Shan'dor powstrzymuje rozrastający się Lightbloom, lecz kolejne uderzenie może przekreślić nad ... (72 characters truncated) ...  strony. Zadaniem jest ich odnaleźć, osłonić i poprowadzić w bezpieczne miejsce przy Runestone, dopóki możliwe jest skutecz